In [1]:

import numpy as np 
import pandas as pd
import os
from PIL import Image
import torch
import  matplotlib.pyplot as plt

DATA_PATH = "E:/ML/UBC"


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [3]:
trainDf = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
allIds = trainDf["image_id"].unique()
trainDf.head()

,image_id,label,image_width,image_height,is_tma
0,4,HGSC,23785,20008,False
1,66,LGSC,48871,48195,False
2,91,HGSC,3388,3388,True
3,281,LGSC,42309,15545,False
4,286,EC,37204,30020,False


In [4]:
trainListIndexed = trainDf.set_index("image_id")

In [5]:
# Only for segmentation data

allIds= next(os.walk(os.path.join(DATA_PATH, "trainProcessed", "segPatchesV2")))[1]
allIds = [int(fid) for fid in allIds]
len(allIds)

152

In [6]:
trainListWang = pd.read_csv(os.path.join(DATA_PATH, "Wang Dataset - all.csv"))
trainListWang.set_index("Image ID", inplace=True)
trainListWang.head()


,No,Patient ID,Treatment effect,Image No.,CA-125 before,CA-125 after,Diagnosis
Image ID,,,,,,,
1612595C,1,2909711,effective,1612595C.svs,91.52,16.72,PSPC (Peritoneal serous papillary carcinoma)
1702510F,2,2959832,effective,1702510F.svs,H>200,29.4,UC
1717035D,3,2983362,effective,1717035D.svs,77.74,27.15,CC
500203D,4,801250,effective,500203D.svs,H>200,18.39,CC
1832548A,5,2188987,effective,1832548A.svs,87.38,8.04,PsC


In [7]:
IMG_SIZE = (384, 384)
eps=1e-12

def readImage(path, skipResize=False):
    data = Image.open(path).convert("RGB")
        
    w, h = data.width, data.height
    # centerWindow = data[w//4:3*w//4, h//4:3*h//4]
    # medValue = np.median(data)

    #Center crop
    data = np.array(data)
    if w>h:
        diff = w-h
        data = data[diff//2:diff//2+h, :, :]
    if h>w:
        diff = h-w
        data = data[:, diff//2:diff//2+w, :]

    # data = data - np.min(data)
    # data = data * 1.0/(np.max(data)+eps)

    w, h, c = data.shape[0], data.shape[1], data.shape[2]

    # resize
    if not skipResize:
        if not (w == IMG_SIZE[0] and h == IMG_SIZE[1]):
            data = np.array(Image.fromarray(data).resize(IMG_SIZE))
    
    # data = data/(np.max(data)+eps) * 2 - 1

    # data = (data * 255).astype(np.uint8)
    return data



In [8]:
from sklearn.preprocessing import LabelEncoder
import warnings
import pickle 

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open('./LabelEncoder5.pkl', 'rb') as f:
        enc = pickle.load(f) 
    print(enc.classes_)
    print(enc.transform(["LGSC"]))

['CC' 'EC' 'HGSC' 'LGSC' 'MC']
[3]


In [9]:
# import timm
# from torchvision.models.feature_extraction import get_graph_node_names
# from torchvision.models.feature_extraction import create_feature_extractor
# import torchvision

# model = timm.create_model('edgenext_base', pretrained=True, num_classes=len(enc.classes_))
# print(model)


# train_nodes, eval_nodes = get_graph_node_names(model)
# print(eval_nodes)

# checkpoint = torch.load(os.path.join("./", "edgenextBase_epoch_8.pt"), map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.to(device)
# model.eval()


# return_nodes = {
#     'global_pool.flatten': 'features',
# }

# model = create_feature_extractor(model, return_nodes=return_nodes)

In [10]:
import timm
from torchvision.models.feature_extraction import get_graph_node_names
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision

model = timm.create_model('resnet34d', pretrained=True, num_classes=len(enc.classes_))

checkpoint = torch.load(os.path.join("./", "seg_resnet34d_384_5_epoch_15_ValAcc_0.9726.pt"), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(model)


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ResNet(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
   

In [11]:

activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# h = model.head.flatten.register_forward_hook(get_activation('global_pool'))
h = model.global_pool.register_forward_hook(get_activation('global_pool'))

x = torch.randn((1,3,384,384)).to(device)
output = model(x)
print(activation["global_pool"].shape)
# h.remove()

torch.Size([1, 512])


In [12]:
# testIn = torch.randn((1,3,256,256)).to(device)
# out = model(testIn)
# out["features"].shape

In [27]:
from torchvision.transforms import v2
from tqdm import tqdm

transformsVal = v2.Compose([
    v2.ToDtype(torch.float32),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

X=[]
y=[]
tma=[]

for tid in tqdm(allIds, smoothing=0.05):
    imFolder = os.path.join(DATA_PATH, "trainProcessed\\segPatchesV2", str(tid))
    imArray = []
    targets = trainListIndexed.loc[tid]
    label = np.array([targets["label"]])
    isTMA = np.array([targets["is_tma"]])
    # if not isTMA[0] and np.random.random()<0.95:
    #     continue
    encLabel = enc.transform(label)

    for root, dirs, files in os.walk(imFolder):
        if not "non-cancerous" in root:
            for f in files:
                if f.endswith(".png"):
                    data = readImage(os.path.join(root, f))
                    imArray.append(data)
    imArray = np.swapaxes(np.array(imArray), 1,-1).astype(np.float32)/255.0
    imArray = transformsVal(torch.Tensor(imArray)).to(device)

    featVect=[]
    if imArray.shape[0]>32:
        splits = np.linspace(0,imArray.shape[0], imArray.shape[0]//32, dtype=np.int64)
        for j,spl in enumerate(splits):
            if j == splits.shape[0]-1:
                break
            XSplitted = imArray[spl:splits[j+1]]
            out = model(XSplitted)
            featVectTemp = activation["global_pool"]
            featVect = [*featVect, *featVectTemp.detach().cpu().numpy()]
    else:
        out = model(imArray)
        # featVect = out["features"]
        featVect = activation["global_pool"].detach().cpu().numpy()
        # featVect = torch.max(featVect, 0)
        # X.append(featVect.values.detach().cpu().numpy())
    
    X.append(featVect)
    y.append(encLabel[0])
    tma.append(isTMA[0])


# for tid in tqdm(list(trainListWang.index)):
#     imFolder = os.path.join(DATA_PATH, "PKG-OBR_Patches", str(tid))
#     targets = trainListWang.loc[tid]
#     label = np.array([targets["Diagnosis"]])[0]
#     if pd.isna(label):
#         continue
#     if "UC" in label:
#         label = np.array(["Other"])
#         encLabel = enc.transform(label)
#     elif "CC" in label:
#         label = np.array(["CC"])
#         encLabel = enc.transform(label)
#     elif "EmAC" in label:
#         label = np.array(["EC"])
#         encLabel = enc.transform(label)
#     elif "MC" in label:
#         label = np.array(["MC"])
#         encLabel = enc.transform(label)
#     elif "PSPC" in label or "PsC" in label or "PSC" in label or "Psc" in label:
#         label = np.array(["HGSC"])
#         encLabel = enc.transform(label)
#     else:
#         print("invalid label for PKG-OBR", label)
#     imArray = []
#     for root, dirs, files in os.walk(imFolder):
#         for f in files:
#             if f.endswith(".png"):
#                 data = readImage(os.path.join(root, f))
#                 imArray.append(data)
#     if len(imArray)==0:
#         print("empty array", imFolder)
#         continue
#     imArray = np.swapaxes(np.array(imArray), 1,-1).astype(np.float32)/255.0
#     imArray = transformsVal(torch.Tensor(imArray)).to(device)

#     out = model(imArray)
#     featVect = out["features"]
#     featVect = torch.max(featVect, 0)

#     X.append(featVect.values.detach().cpu().numpy())
#     y.append(encLabel[0])


100%|██████████| 152/152 [21:57<00:00,  8.67s/it]


In [28]:
X[0][1].shape

(512,)

In [29]:
X = np.array(X)
y = np.array(y)
tma=np.array(tma)
X.shape

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (152,) + inhomogeneous part.

In [30]:

with open(os.path.join(DATA_PATH,"./FeatureDataSeg.pkl"), "wb") as f:
        pickle.dump(zip(X,y, tma), f)